# 9.8 多 LoRA 动态服务 (Multi-LoRA Dynamic Serving)

> 🕐 预估学习时间：30分钟

在大规模 LLM 服务中，不同用户/任务往往需要不同的微调能力。为每个任务部署独立的全量微调模型代价高昂：显存占用线性增长、加载延迟高、运维复杂。

多 LoRA 动态服务通过共享一个基座模型 + 多个轻量 LoRA 适配器，实现按需切换、批量推理、多租户隔离，大幅降低服务成本。

本节涵盖：
- 多 LoRA 服务的动机与成本模型
- LoRA 动态切换（hot-swapping）
- S-LoRA：批量多 LoRA 推理
- Punica 与适配器管理
- 多租户服务架构实践

代表系统：S-LoRA、Punica、LoRAX、vLLM (LoRA support)。

## 1. 多 LoRA 服务概述

**为什么需要多 LoRA？**

实际生产中，一个基座模型（如 Llama-7B）需要服务成百上千个不同任务：
- **垂直领域**：医疗、法律、金融、客服等不同领域适配器
- **个性化**：每个用户的偏好适配器
- **A/B 实验**：同时测试多个微调版本
- **多语言**：不同语言的翻译/生成适配器

**成本对比（以 7B 模型 + 100 个任务为例）**：
- 独立部署：100 × 14GB ≈ 1.4TB 显存（不可行）
- 多 LoRA：14GB 基座 + 100 × 20MB ≈ 16GB（可行）

**核心收益**：
- 显存占用从 O(N × M) 降为 O(M + N × r)
- 加载/切换延迟从分钟级降为毫秒级
- 运维复杂度大幅降低

**关键技术挑战**：
- 适配器切换开销
- 批量推理时不同 LoRA 的混合计算
- 适配器缓存与淘汰策略
- 多租户隔离与调度

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

torch.manual_seed(42)


class TinyTransformer(nn.Module):
    '''简化 Transformer，用于演示多 LoRA 服务。'''
    def __init__(self, vocab=1000, d_model=128, n_heads=4, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, batch_first=True)
            for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_model, vocab, bias=False)

    def forward(self, x):
        h = self.embed(x)
        for layer in self.layers:
            h = layer(h)
        return self.head(h)


class SeparateModelServer:
    '''基线：每个任务部署独立微调模型。'''
    def __init__(self, n_tasks=5, vocab=1000, d_model=128):
        self.n_tasks = n_tasks
        self.models = [TinyTransformer(vocab, d_model) for _ in range(n_tasks)]
        # 模拟每个模型独立微调
        for m in self.models:
            for p in m.parameters():
                p.data += torch.randn_like(p) * 0.02
        self.memory_bytes = sum(
            sum(p.numel() * p.element_size() for p in m.parameters())
            for m in self.models
        )

    def serve(self, task_id, token_ids):
        '''为指定任务推理。'''
        with torch.no_grad():
            return self.models[task_id](token_ids)


# 模拟 5 个任务的服务
server = SeparateModelServer(n_tasks=5)
sample = torch.randint(0, 1000, (1, 16))

print('=== 基线：独立模型部署 ===')
print(f'任务数: {server.n_tasks}')
print(f'总显存占用: {server.memory_bytes / 1024 / 1024:.2f} MB')
print(f'平均每任务: {server.memory_bytes / server.n_tasks / 1024 / 1024:.2f} MB')

# 测量推理延迟
start = time.perf_counter()
for _ in range(20):
    for t in range(server.n_tasks):
        _ = server.serve(t, sample)
elapsed = time.perf_counter() - start

print(f'20 轮推理总耗时: {elapsed*1000:.2f} ms')
print(f'\nKey: 独立部署下显存随任务数线性增长，N=100 时约需 {server.memory_bytes / server.n_tasks * 100 / 1024 / 1024:.1f} MB，不可行。')
print('多 LoRA 服务的目标：共享基座，按需加载适配器。')

## 2. LoRA 动态切换

**LoRA 回顾**：将权重更新分解为低秩矩阵 ΔW = B @ A，其中 A ∈ R^(r×d)，B ∈ R^(d×r)，r 远小于 d。推理时 h' = Wx + B(Ax)。

**动态切换的核心思想**：
- 基座模型权重 W 常驻显存（只加载一次）
- 每个任务的 LoRA 参数 (A_i, B_i) 体积小（MB 级），可快速切换
- 切换方式：
  1. **在线合并**：W' = W + B_i @ A_i，每次切换重新合并
  2. **运行时计算**：h' = Wx + B_i(A_i x)，不修改 W
  3. **预合并缓存**：对热点适配器预合并并缓存

**切换开销分析**：
- LoRA 参数加载：~MB 级，PCIe 带宽下毫秒级
- 在线合并：O(d²) 计算，对 7B 模型约几十毫秒
- 运行时计算：O(r × d) per token，开销极小

**适用场景**：
- 适配器数量较少（< 100）
- 请求粒度切换
- 单请求单 LoRA

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

torch.manual_seed(42)


class LoRALayer(nn.Module):
    '''单个 LoRA 适配器层。'''
    def __init__(self, d_model, r=8):
        super().__init__()
        self.A = nn.Parameter(torch.randn(r, d_model) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_model, r))

    def forward(self, x):
        return x + (x @ self.A.t()) @ self.B


class DynamicLoRAServer:
    '''动态 LoRA 服务：基座共享 + 按需切换适配器。'''
    def __init__(self, n_adapters=5, d_model=128, r=8):
        self.d_model = d_model
        self.r = r
        # 基座模型（只加载一次）
        self.base_linear = nn.Linear(d_model, d_model, bias=False)
        # 多个 LoRA 适配器
        self.adapters = {
            i: {
                'A': torch.randn(r, d_model) * 0.01,
                'B': torch.randn(d_model, r) * 0.01,
            }
            for i in range(n_adapters)
        }
        self.current_adapter = None
        self.switch_count = 0

    def load_adapter(self, adapter_id):
        '''加载指定适配器到显存（这里只是切换引用）。'''
        if self.current_adapter != adapter_id:
            self.current_adapter = adapter_id
            self.switch_count += 1

    def forward(self, adapter_id, x):
        self.load_adapter(adapter_id)
        adapter = self.adapters[adapter_id]
        # 基座计算 + LoRA delta
        base_out = self.base_linear(x)
        lora_delta = (x @ adapter['A'].t()) @ adapter['B']
        return base_out + lora_delta

    def memory_usage(self):
        base_mem = self.base_linear.weight.numel() * 4
        adapter_mem = sum(
            (a['A'].numel() + a['B'].numel()) * 4
            for a in self.adapters.values()
        )
        return base_mem, adapter_mem


# 对比：独立模型 vs 动态 LoRA
n_tasks = 5
d_model = 128

# 独立模型内存（5 个 Linear 层）
separate_mem = n_tasks * d_model * d_model * 4

server = DynamicLoRAServer(n_adapters=n_tasks, d_model=d_model, r=8)
base_mem, adapter_mem = server.memory_usage()
total_lora_mem = base_mem + adapter_mem

print('=== 动态 LoRA 切换 ===')
print(f'任务数: {n_tasks}, d_model: {d_model}, LoRA rank: 8')
print(f'\n独立部署显存: {separate_mem / 1024:.2f} KB')
print(f'多 LoRA 显存: {total_lora_mem / 1024:.2f} KB (基座 {base_mem/1024:.2f} + 适配器 {adapter_mem/1024:.2f})')
print(f'节省比例: {(1 - total_lora_mem / separate_mem) * 100:.1f}%')

# 演示切换
x = torch.randn(2, 8, d_model)
start = time.perf_counter()
for _ in range(100):
    for t in range(n_tasks):
        _ = server.forward(t, x)
elapsed = time.perf_counter() - start

print(f'\n切换次数: {server.switch_count}')
print(f'100 轮推理耗时: {elapsed*1000:.2f} ms')
print(f'\nKey: 多 LoRA 显存仅 {total_lora_mem/separate_mem*100:.1f}% 于独立部署，rank 越小节省越多。')
print('切换开销极低，适合请求粒度的适配器路由。')

## 3. S-LoRA

**S-LoRA** (Serving LoRA) 是一种高效的多 LoRA 批量推理系统，核心创新：

**1. 统一批处理**：
- 同一批次可包含使用不同 LoRA 的请求
- 基座计算一次（batch 共享），LoRA delta 分别计算
- 避免为每个 LoRA 单独跑前向

**2. 自定义 CUDA Kernel**：
- 将 LoRA 的 A、B 矩阵按 batch 维度分组
- 使用 BGMV（Batched GEMV）kernel 一次性计算所有请求的 delta
- 避免 Python 循环和 kernel launch 开销

**3. 适配器分页**：
- LoRA 参数以分页方式存储在 GPU 显存
- 支持远超显存容量的适配器集合

**计算流程**：
```
对于 batch 中每个请求 i（使用 LoRA_i）:
  base_out_i = W @ x_i           # 共享基座
  delta_i = B_i @ (A_i @ x_i)    # 各自 LoRA
  out_i = base_out_i + delta_i
```

**性能**：在 A100 上可同时服务数千个 LoRA，吞吐量相比独立部署提升 10x+。

**与普通动态切换的区别**：
- 动态切换：同一 batch 内只能用一个 LoRA
- S-LoRA：同一 batch 内可混合多个 LoRA

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

torch.manual_seed(42)


class BatchedLoRACompute(nn.Module):
    '''批量 LoRA 计算：一次前向处理多个不同 LoRA 的请求。'''
    def __init__(self, n_adapters, d_model, r=8):
        super().__init__()
        self.n_adapters = n_adapters
        self.d_model = d_model
        self.r = r
        # 将所有 LoRA 的 A、B 堆叠为 [n_adapters, ...]
        self.A = torch.randn(n_adapters, r, d_model) * 0.01
        self.B = torch.randn(n_adapters, d_model, r) * 0.01

    def forward(self, x, adapter_ids):
        '''
        x: [batch, ..., d_model]
        adapter_ids: [batch] 每个样本使用的适配器 id
        '''
        # 取出每个样本对应的 A、B
        A_batch = self.A[adapter_ids]  # [batch, r, d_model]
        B_batch = self.B[adapter_ids]  # [batch, d_model, r]
        # 计算 delta = (x @ A^T) @ B
        # delta = x @ A^T -> [batch, ..., r]
        delta = torch.einsum('b...d,brd->b...r', x, A_batch)
        # delta @ B -> [batch, ..., d_model]
        delta = torch.einsum('b...r,bdr->b...d', delta, B_batch)
        return delta


class SLoRAServer:
    '''S-LoRA 风格服务：批量混合多 LoRA 推理。'''
    def __init__(self, n_adapters=5, d_model=128, r=8):
        self.d_model = d_model
        self.base_linear = nn.Linear(d_model, d_model, bias=False)
        self.batched_lora = BatchedLoRACompute(n_adapters, d_model, r)
        self.n_adapters = n_adapters

    def serve_batch(self, x, adapter_ids):
        '''
        x: [batch, seq, d_model]
        adapter_ids: [batch]
        '''
        base_out = self.base_linear(x)
        delta = self.batched_lora(x, adapter_ids)
        return base_out + delta


# 演示：同一 batch 内混合不同 LoRA 请求
n_adapters = 5
server = SLoRAServer(n_adapters=n_adapters, d_model=128, r=8)

# 构造混合 batch：8 个请求，每个随机使用一个 LoRA
batch_size = 8
seq_len = 16
x = torch.randn(batch_size, seq_len, 128)
adapter_ids = torch.randint(0, n_adapters, (batch_size,))

print('=== S-LoRA 批量混合推理 ===')
print(f'适配器数: {n_adapters}, batch size: {batch_size}')
print(f'本 batch 各请求使用的 LoRA: {adapter_ids.tolist()}')

# 批量推理
start = time.perf_counter()
for _ in range(100):
    out = server.serve_batch(x, adapter_ids)
batched_time = time.perf_counter() - start

# 对比：逐请求处理
start = time.perf_counter()
for _ in range(100):
    outs = []
    for i in range(batch_size):
        o = server.serve_batch(x[i:i+1], adapter_ids[i:i+1])
        outs.append(o)
sequential_time = time.perf_counter() - start

print(f'\n批量推理 100 轮: {batched_time*1000:.2f} ms')
print(f'逐请求推理 100 轮: {sequential_time*1000:.2f} ms')
print(f'加速比: {sequential_time / batched_time:.2f}x')
print(f'\nKey: S-LoRA 通过批量 GEMV kernel 让同一 batch 内不同 LoRA 共享基座计算，吞吐显著提升。')
print('核心是 einsum 一次性计算所有样本的 delta，避免 Python 循环。')

## 4. Punica 与适配器管理

**Punica** 是最早的多租户 LoRA 服务系统之一，核心贡献：

**1. Multi-Tenant Batched Inference**：
- 多个租户（tenant）共享同一基座模型
- 不同租户的请求可在同一 batch 内处理
- 通过 SGMV（Segmented GEMV）kernel 高效计算

**2. 适配器调度**：
- 当适配器数量超过显存容量时，需要缓存与淘汰
- LRU（最近最少使用）策略：淘汰最久未访问的适配器
- 预取（prefetch）：根据请求预测提前加载适配器

**3. 适配器生命周期**：
- 注册（register）：新适配器加入系统
- 加载（load）：从 CPU/SSD 拷贝到 GPU
- 激活（activate）：设为当前可用
- 驱逐（evict）：从 GPU 移除

**适配器管理挑战**：
- 显存预算有限，需在基座、KV cache、适配器间分配
- 适配器访问模式可能不均匀（少数热点，大量冷门）
- 突发流量下需快速响应

**调度策略**：
- **FIFO**：按到达顺序处理（简单但不优化复用）
- **Adapter-Aware**：优先处理与当前已加载适配器匹配的请求
- **Batching**：聚合相同适配器的请求一起处理

In [ ]:
import torch
import torch.nn as nn
import math
import time
from collections import OrderedDict, deque
import random

torch.manual_seed(42)


class AdapterManager:
    '''适配器管理：LRU 缓存 + 加载/驱逐。'''
    def __init__(self, n_total=30, gpu_capacity=8, d_model=128, r=8):
        self.n_total = n_total
        self.gpu_capacity = gpu_capacity
        self.d_model = d_model
        self.r = r
        # 所有适配器存在 CPU（模拟）
        self.cpu_adapters = {
            i: {
                'A': torch.randn(r, d_model) * 0.01,
                'B': torch.randn(d_model, r) * 0.01,
            }
            for i in range(n_total)
        }
        # GPU 上的 LRU 缓存
        self.gpu_cache = OrderedDict()
        self.load_count = 0
        self.evict_count = 0
        self.hit_count = 0
        self.miss_count = 0

    def get(self, adapter_id):
        '''获取适配器，命中则直接返回，未命中则加载。'''
        if adapter_id in self.gpu_cache:
            self.hit_count += 1
            self.gpu_cache.move_to_end(adapter_id)
            return self.gpu_cache[adapter_id]
        self.miss_count += 1
        self._load(adapter_id)
        return self.gpu_cache[adapter_id]

    def _load(self, adapter_id):
        '''从 CPU 加载适配器到 GPU，必要时驱逐。'''
        while len(self.gpu_cache) >= self.gpu_capacity:
            evicted_id, _ = self.gpu_cache.popitem(last=False)
            self.evict_count += 1
        self.gpu_cache[adapter_id] = self.cpu_adapters[adapter_id]
        self.load_count += 1

    def stats(self):
        total = self.hit_count + self.miss_count
        hit_rate = self.hit_count / total if total > 0 else 0
        return {
            'hits': self.hit_count,
            'misses': self.miss_count,
            'hit_rate': hit_rate,
            'loads': self.load_count,
            'evictions': self.evict_count,
        }


class RequestScheduler:
    '''请求调度：优化适配器复用。'''
    def __init__(self, adapter_manager):
        self.manager = adapter_manager
        self.queue = deque()

    def submit(self, request):
        self.queue.append(request)

    def schedule_fifo(self):
        '''FIFO 调度。'''
        order = list(self.queue)
        self.queue.clear()
        return order

    def schedule_adapter_aware(self):
        '''适配器感知调度：按 adapter_id 分组。'''
        order = sorted(self.queue, key=lambda r: r['adapter_id'])
        self.queue.clear()
        return order


# 模拟 30 个适配器，GPU 只能容纳 8 个
manager = AdapterManager(n_total=30, gpu_capacity=8)
scheduler = RequestScheduler(manager)

# 生成 200 个请求，访问模式：少数热点 + 大量冷门
random.seed(42)
requests = []
for _ in range(200):
    # 80% 请求访问前 5 个热点适配器
    if random.random() < 0.8:
        aid = random.randint(0, 4)
    else:
        aid = random.randint(5, 29)
    requests.append({'adapter_id': aid, 'data': torch.randn(1, 8, 128)})

print('=== 适配器管理与调度 ===')
print(f'总适配器数: {manager.n_total}, GPU 容量: {manager.gpu_capacity}')
print(f'请求数: {len(requests)}')

# 测试 FIFO 调度
manager_fifo = AdapterManager(n_total=30, gpu_capacity=8)
for req in requests:
    manager_fifo.get(req['adapter_id'])
stats_fifo = manager_fifo.stats()

# 测试适配器感知调度（按 adapter_id 排序）
manager_aa = AdapterManager(n_total=30, gpu_capacity=8)
sorted_reqs = sorted(requests, key=lambda r: r['adapter_id'])
for req in sorted_reqs:
    manager_aa.get(req['adapter_id'])
stats_aa = manager_aa.stats()

# 提取指标到局部变量，便于 f-string 输出
fifo_hits = stats_fifo['hits']
fifo_misses = stats_fifo['misses']
fifo_hit_rate = stats_fifo['hit_rate'] * 100
fifo_loads = stats_fifo['loads']
fifo_evictions = stats_fifo['evictions']
aa_hits = stats_aa['hits']
aa_misses = stats_aa['misses']
aa_hit_rate = stats_aa['hit_rate'] * 100
aa_loads = stats_aa['loads']
aa_evictions = stats_aa['evictions']

print(f'\n--- FIFO 调度 ---')
print(f'命中: {fifo_hits}, 未命中: {fifo_misses}')
print(f'命中率: {fifo_hit_rate:.1f}%')
print(f'加载次数: {fifo_loads}, 驱逐次数: {fifo_evictions}')

print(f'\n--- 适配器感知调度 ---')
print(f'命中: {aa_hits}, 未命中: {aa_misses}')
print(f'命中率: {aa_hit_rate:.1f}%')
print(f'加载次数: {aa_loads}, 驱逐次数: {aa_evictions}')

print(f'\nKey: 适配器感知调度将相同 LoRA 的请求聚合，命中率从 {fifo_hit_rate:.1f}% 提升到 {aa_hit_rate:.1f}%。')
print('LRU + 智能调度是多 LoRA 服务控制显存压力的关键。')

## 5. 实践：多租户服务架构

**多租户架构**：多个用户/组织（租户）共享同一推理服务，每个租户有自己的 LoRA 适配器。

**架构组件**：
1. **API Gateway**：接收请求，识别租户
2. **Router**：根据租户 ID 路由到对应适配器
3. **Adapter Pool**：管理所有租户的适配器
4. **Inference Engine**：共享基座 + 动态 LoRA
5. **Scheduler**：批量聚合、优先级调度

**租户隔离**：
- 逻辑隔离：不同租户的请求使用不同 LoRA
- 性能隔离：QPS 限流、配额管理
- 数据隔离：租户间 KV cache 不共享

**生产实践要点**：
- **冷启动**：新租户适配器首次加载延迟
- **预热**：热点租户适配器常驻 GPU
- **降级**：显存不足时回退到基座模型
- **监控**：每租户 QPS、延迟、错误率

**典型部署**：
- 单 GPU 服务 50-200 个租户
- 多 GPU 横向扩展，按租户分片
- 适配器存储在分布式文件系统

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
from collections import OrderedDict, defaultdict
import random

torch.manual_seed(42)


class MultiTenantServer:
    '''多租户 LoRA 服务：按租户路由，共享基座。'''
    def __init__(self, n_tenants=20, d_model=128, r=8, gpu_capacity=10):
        self.d_model = d_model
        self.base_linear = nn.Linear(d_model, d_model, bias=False)
        self.n_tenants = n_tenants
        # 每个租户一个适配器
        self.adapters = {
            i: {
                'A': torch.randn(r, d_model) * 0.01,
                'B': torch.randn(d_model, r) * 0.01,
            }
            for i in range(n_tenants)
        }
        # LRU 缓存
        self.gpu_cache = OrderedDict()
        self.gpu_capacity = gpu_capacity
        # 指标
        self.metrics = defaultdict(list)
        self.total_requests = 0
        self.cache_hits = 0

    def _get_adapter(self, tenant_id):
        if tenant_id in self.gpu_cache:
            self.cache_hits += 1
            self.gpu_cache.move_to_end(tenant_id)
            return self.gpu_cache[tenant_id]
        while len(self.gpu_cache) >= self.gpu_capacity:
            self.gpu_cache.popitem(last=False)
        self.gpu_cache[tenant_id] = self.adapters[tenant_id]
        return self.gpu_cache[tenant_id]

    def serve(self, tenant_id, x):
        start = time.perf_counter()
        adapter = self._get_adapter(tenant_id)
        base_out = self.base_linear(x)
        delta = (x @ adapter['A'].t()) @ adapter['B']
        out = base_out + delta
        latency = (time.perf_counter() - start) * 1000
        self.metrics[tenant_id].append(latency)
        self.total_requests += 1
        return out

    def serve_batch(self, requests):
        '''批量服务混合租户请求（S-LoRA 风格）。'''
        start = time.perf_counter()
        # 按 tenant 分组
        groups = defaultdict(list)
        for i, req in enumerate(requests):
            groups[req['tenant_id']].append(i)
        results = [None] * len(requests)
        for tenant_id, indices in groups.items():
            adapter = self._get_adapter(tenant_id)
            xs = torch.stack([requests[i]['data'] for i in indices])
            base_out = self.base_linear(xs)
            delta = (xs @ adapter['A'].t()) @ adapter['B']
            out = base_out + delta
            for j, idx in enumerate(indices):
                results[idx] = out[j]
        latency = (time.perf_counter() - start) * 1000
        self.total_requests += len(requests)
        return results, latency

    def stats(self):
        all_latencies = [l for lats in self.metrics.values() for l in lats]
        return {
            'total_requests': self.total_requests,
            'cache_hits': self.cache_hits,
            'hit_rate': self.cache_hits / max(self.total_requests, 1),
            'avg_latency': sum(all_latencies) / max(len(all_latencies), 1),
            'p99_latency': sorted(all_latencies)[int(len(all_latencies) * 0.99)] if all_latencies else 0,
        }


# 模拟多租户工作负载
server = MultiTenantServer(n_tenants=20, d_model=128, r=8, gpu_capacity=10)
random.seed(42)

# 生成 500 个请求，租户访问分布不均（Zipf-like）
tenant_weights = [10 / (i + 1) for i in range(20)]
requests = []
for _ in range(500):
    tenant_id = random.choices(range(20), weights=tenant_weights)[0]
    requests.append({
        'tenant_id': tenant_id,
        'data': torch.randn(1, 8, 128),
    })

print('=== 多租户 LoRA 服务 ===')
print(f'租户数: {server.n_tenants}, GPU 缓存容量: {server.gpu_capacity}')
print(f'总请求数: {len(requests)}')

# 单请求服务
for req in requests:
    server.serve(req['tenant_id'], req['data'])
stats = server.stats()

# 提取指标到局部变量，便于 f-string 输出
total_req = stats['total_requests']
hit_rate = stats['hit_rate'] * 100
avg_latency = stats['avg_latency']
p99_latency = stats['p99_latency']

print(f'\n--- 单请求模式 ---')
print(f'总请求: {total_req}')
print(f'缓存命中率: {hit_rate:.1f}%')
print(f'平均延迟: {avg_latency:.3f} ms')
print(f'P99 延迟: {p99_latency:.3f} ms')

# 批量服务
server2 = MultiTenantServer(n_tenants=20, d_model=128, r=8, gpu_capacity=10)
batch_size = 32
total_latency = 0
for i in range(0, len(requests), batch_size):
    batch = requests[i:i+batch_size]
    _, lat = server2.serve_batch(batch)
    total_latency += lat
stats2 = server2.stats()

# 提取批量模式指标
total_req2 = stats2['total_requests']
hit_rate2 = stats2['hit_rate'] * 100
avg_per_req = total_latency / len(requests)

print(f'\n--- 批量模式 (batch={batch_size}) ---')
print(f'总请求: {total_req2}')
print(f'缓存命中率: {hit_rate2:.1f}%')
print(f'总批量延迟: {total_latency:.2f} ms')
print(f'平均每请求: {avg_per_req:.3f} ms')

print(f'\nKey: 多租户服务通过共享基座 + LRU 适配器缓存，用 {server.gpu_capacity} 个槽位服务 {server.n_tenants} 个租户。')
print(f'批量模式进一步降低平均延迟（{avg_per_req:.3f}ms vs 单请求 {avg_latency:.3f}ms）。')

## 📝 课后思考题

1. 多 LoRA 服务相比独立部署全量微调模型，在显存、延迟、吞吐三个维度上各有什么优势？什么场景下多 LoRA 反而不如独立部署？

2. S-LoRA 的批量 GEMV kernel 相比朴素的逐请求 LoRA 计算，为什么能显著提升吞吐？请从 kernel launch、内存带宽、并行度三个角度分析。

3. 当适配器数量远超 GPU 显存容量时，LRU 淘汰策略可能存在什么问题？请设计一种更优的缓存策略（考虑访问模式、预取、优先级）。

4. 在多租户场景下，如何保证不同租户之间的性能隔离？如果一个租户突发大量请求，应该如何处理以避免影响其他租户？